# Modelos no lineales (1): Threshold VAR y el Fenómeno El Niño

Hasta la sesión 3 los coeficientes eran fijos: un choque se propaga igual en 1998 que en 2019, en
un año normal que en medio de un Niño costero. Este notebook rompe ese supuesto con un
**Threshold VAR (TVAR)**: dos VAR, uno por régimen, y una variable observable que decide en qué
régimen está la economía.

La variable que decide es la **anomalía de la temperatura superficial del mar en la región Niño
1+2** (frente a la costa de Perú y Ecuador, NOAA), $F_t$. Y hay una segunda idea: la temperatura
del mar no depende de la economía peruana. Ese supuesto se impone con **exogeneidad de bloque**.
La anomalía forma un bloque externo, un AR(2) lineal que no cambia de régimen, y las variables
domésticas $\mathbf{x}_t$ forman un bloque con dos regímenes que responde a la anomalía corriente y
rezagada:

$$
F_t = a + \phi_1 F_{t-1} + \phi_2 F_{t-2} + u^F_t,
$$

$$
\mathbf{x}_t = \Big(c_1 + \sum_{j=1}^{2} B_{1,j}\mathbf{x}_{t-j} + \sum_{j=0}^{2}\gamma_{1,j}F_{t-j} + \mathbf{u}_{1,t}\Big)S_t
 + \Big(c_2 + \sum_{j=1}^{2} B_{2,j}\mathbf{x}_{t-j} + \sum_{j=0}^{2}\gamma_{2,j}F_{t-j} + \mathbf{u}_{2,t}\Big)(1 - S_t),
\qquad \mathbf{u}_{i,t} \sim N(0, \Sigma_i),
$$

$$
S_t = 1 \iff F_{t-d} \le Z^*,
$$

con el umbral $Z^*$ y el rezago $d$ estimados junto con los coeficientes. El bloque doméstico es
$\mathbf{x}_t = (y_t,\ \pi_t,\ ing_t)'$: el crecimiento interanual del PBI, la inflación interanual
y el crecimiento interanual de los **ingresos corrientes del gobierno central** en términos reales,
el canal por el que El Niño llega a las finanzas públicas. La muestra es trimestral, de 1996Q1 a
2026Q2, con la pandemia tratada como en la sesión 2: escalas de volatilidad de Lenza y Primiceri.

1. **Datos**: la anomalía, la actividad, los precios y los ingresos fiscales.
2. **Regímenes y exogeneidad de bloque**: umbral, rezago, trimestres Niño y dos factores de Bayes.
3. **Respuestas por régimen**: un calentamiento de 1 °C, con y sin exogeneidad de bloque.
4. **Escenarios**: el Niño de 2026.
5. **Robustez**: controles, muestras e índices.


## 1. Datos

La base del curso trae las anomalías de NOAA (`nino12` y `nino34`, promedios trimestrales de la
serie mensual, en grados respecto al promedio de 1981-2010) y las series del BCRP: el PBI, el IPC y
los ingresos corrientes reales del gobierno central (`icgc`, serie `PN38689FM`, en millones de
soles de diciembre de 2021). Los ingresos empiezan en 1995, de modo que las tasas interanuales
empiezan en 1996Q1.

In [ ]:
import os
for var in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "OMP_NUM_THREADS"):
    os.environ.setdefault(var, "1")         # antes de numpy: con matrices pequeñas, un hilo rinde más que varios
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from MacroPy import ThresholdVAR, BayesianVAR, generate_series_plot, set_style
from MacroPy.plots_uc import plot_fan_chart
import sys; sys.path.append("..")          # course_data.py vive en la raíz del repositorio
import course_data as cd

set_style()
levels = cd.load()                          # copia congelada de la base del curso
g = cd.yoy(levels[["pbi", "ipc", "icgc"]]).rename(columns={"pbi": "y", "ipc": "pi", "icgc": "ing"})
Y = levels[["nino12"]].join(g).loc["1996-03-01":"2026-06-01"].dropna()
names = list(Y.columns)
titles = ["Anomalía Niño 1+2 (°C)", "PBI (var. % interanual)", "Inflación (var. % interanual)",
          "Ingresos fiscales (var. % interanual)"]
print(f"Muestra: {cd.quarter(Y.index[0])} a {cd.quarter(Y.index[-1])} ({len(Y)} trimestres)")
generate_series_plot(Y, series_titles=titles, title="El Niño costero, la actividad y los ingresos fiscales")

In [ ]:
nino = levels["nino12"].dropna().loc["1980-03-01":]
fuertes = nino[nino > 1.5]
print("Trimestres con anomalía mayor a 1.5 °C desde 1980:")
print(", ".join(cd.quarter(d) for d in fuertes.index))
print("\nPBI e ingresos fiscales alrededor de la pandemia (var. % interanual):")
Y.loc["2019-12-01":"2022-03-01", ["y", "ing"]].T.rename(columns=cd.quarter).round(1)

Cinco episodios fuertes desde 1980: 1982-83, 1997-98, 2015-16, 2023 y el que está en curso en
2026. El de 2017, el Niño costero de marzo, fue intenso pero corto y queda por debajo de 1.5 °C en
el promedio trimestral. La muestra contiene tres, 1997-98, 2015-16 y 2023-24, y la sección 4 usa
el modelo para el de 2026.

La muestra contiene también la pandemia: el PBI cae 30 % interanual en 2020Q2 y, por la aritmética
de la tasa interanual, sube 42 % un año después; los ingresos fiscales caen 39 % y suben 72 %. Son
observaciones diez veces más grandes que cualquier recesión asociada a un Niño, y entran también
como **rezagos** de los trimestres siguientes. Sin tratamiento, el segundo régimen del TVAR se
dedicaría a ellas (sección 2). El tratamiento es el de la sesión 2: **escalas de volatilidad de
Lenza y Primiceri (2022)**, $s_t \ge 1$ para cada trimestre de 2020Q1 a 2021Q4, que dividen esas
filas antes de estimar; la ventana se extiende dos trimestres más allá del rebote para que los
rezagos contaminados también pesen menos. Las escalas se estiman por verosimilitud marginal en el
VAR lineal con el mismo prior, antes de mirar ningún régimen, y como la tasa interanual pone el
segundo pico en 2021Q2, las ocho son libres (`covid_n_free=8`).

## 2. Regímenes y exogeneidad de bloque

`b_exo` usa la convención de `BayesianVAR`: `b_exo[i, j] = 1` excluye la variable `j` de la
ecuación `i`. Aquí la fila de la anomalía excluye al PBI, la inflación y los ingresos. En
`ThresholdVAR` la restricción es una estructura de bloques exacta, no un prior muy ajustado: el
prior de observaciones ficticias tiene covarianza de Kronecker, que no admite exclusiones ecuación
por ecuación, así que la anomalía se estima como un BVAR lineal propio y el bloque doméstico como
una regresión con umbral sobre sus rezagos y sobre $F_t$, $F_{t-1}$ y $F_{t-2}$. Los coeficientes de
la anomalía corriente, $\gamma_{i,0}$, llevan su propio prior (`prior_params["lamda0P"]`, uno por
defecto).

In [ ]:
b_exo = np.zeros((4, 4), dtype=int)
b_exo[0, 1:] = 1                 # fila = ecuación: la de la anomalía excluye PBI, inflación e ingresos
window = ("2020Q1", "2021Q4")
N_JOBS = min(4, os.cpu_count() or 1)   # procesos para las respuestas generalizadas, unos 200 MB cada uno
model = ThresholdVAR(Y, threshold_var="nino12", lags=2, max_delay=4, post_draws=4000, burnin=0.5,
                     hor=12, irf_1std=0, seed=1, b_exo=b_exo,
                     covid_mode="lenza-primiceri", covid_window=window, covid_n_free=8)
print("Escalas de volatilidad estimadas, s_t (las filas se dividen por s_t):")
display(pd.DataFrame({cd.quarter(d): [s] for d, s in zip(model.covid_dates, model.covid_scales)},
                     index=["s_t"]).round(1))
model.model_summary()
draws = model.sample_posterior()

Las escalas dicen dónde está la pandemia en estos datos: 2020Q2 (la caída), 2021Q2 (el rebote de
la tasa interanual) y, con menos fuerza, los trimestres en los que esas dos observaciones son
rezagos, 2020Q3 y 2021Q3. En 2020Q4, 2021Q1 y 2021Q4 la escala es casi uno: nada que tratar.

In [ ]:
display(model.threshold_summary().round(2))
sp = model.split_posterior
print("Posterior del rezago d:", sp.groupby("delay")["probability"].sum().round(2).to_dict())
tar, delay = draws["tar"], draws["delay"]
secundaria = tar < 0.5
print(f"Extracciones con umbral menor a 0.5 °C: {secundaria.mean():.1%}; "
      f"su umbral mediano: {np.median(tar[secundaria]):.2f} °C; "
      f"rezago más frecuente: {np.bincount(delay[secundaria]).argmax()}")
lo, hi = np.percentile(tar[~secundaria], [5, 95])
print(f"Moda principal: umbral mediano {np.median(tar[~secundaria]):.2f} °C, intervalo al 90 % {lo:.2f} a {hi:.2f} °C")

El umbral está cerca de **1.2 °C** y el rezago más probable es de **tres trimestres** (cuatro con
probabilidad 0.27). La posterior del umbral es **bimodal**: el 13 % de las extracciones, casi todas
con rezago de cuatro trimestres, lo ubica cerca de $-0.85$ °C, donde casi todos los trimestres
serían "Niño". Por eso el percentil 5 es negativo; en la moda principal el intervalo al 90 % va de
1.0 a 1.7 °C.

¿Qué significa el rezago? Que la **regla de clasificación** mira la anomalía de tres trimestres
antes, no que el daño tarde tres trimestres en llegar: el efecto contemporáneo de la temperatura
entra en ambos regímenes por $\gamma_{i,0}$. El régimen dice en qué estado propaga la economía los
choques; la temperatura de hoy la mueve en cualquiera de los dos.

In [ ]:
model.plot_regimes(regime_label="Régimen Niño", prob_label="P(Niño)",
                   var_label="Anomalía Niño 1+2 (°C)", title="Regímenes El Niño, 1996-2026")
p_nino = model.regime_probability()
print("Trimestres en régimen Niño (P > 0.5):", ", ".join(cd.quarter(d) for d in p_nino.index[p_nino > 0.5]))

Tres episodios: **1998Q1 a 1999Q1**, después del Niño extraordinario de 1997-98; **2016**, después
del de 2015-16 (más 2017Q4); y **2024Q1 a 2024Q3**, después del Niño costero de 2023. La línea
punteada es el umbral; la curva, la probabilidad posterior del régimen Niño, alta en esos trimestres.
El piso cercano a 0.13 en el resto de la muestra es la moda secundaria del umbral.

¿Vale la pena el segundo régimen? ¿Y el bloque? Con exogeneidad de bloque, `log_marginal_likelihood`
devuelve el bloque externo, el bloque doméstico con umbral y con un solo régimen, los totales del
sistema y el factor de Bayes del umbral, que es la diferencia en el bloque doméstico (el externo es
el mismo en ambos modelos y se cancela). Los totales son densidades de los mismos datos que el TVAR
sin bloque, así que también permiten probar la restricción.

In [ ]:
ml = model.log_marginal_likelihood()
joint = ThresholdVAR(Y, threshold_var="nino12", lags=2, max_delay=4, post_draws=4000, burnin=0.5,
                     hor=12, irf_1std=0, seed=1,
                     covid_mode="lenza-primiceri", covid_window=window, covid_n_free=8)
joint.sample_posterior(show_progress=False)
ml_joint = joint.log_marginal_likelihood()
display(ml.round(1).to_frame("con bloque"))
print(f"log factor de Bayes del bloque (sistema con bloque contra sistema conjunto): "
      f"{ml['TVAR'] - ml_joint['TVAR']:.1f}")

Dos factores de Bayes, dos respuestas distintas.

* **El bloque.** El log factor de Bayes de la exogeneidad de bloque es **+3.2**: los datos prefieren
  que la temperatura del mar no responda a la economía peruana. Es lo que uno espera de un fenómeno
  oceánico, y aquí además se verifica.
* **El umbral.** El del segundo régimen es **$-5.2$**, desfavorable al TVAR. Es una lección y no un
  detalle: con trece trimestres en el régimen Niño y diez coeficientes por ecuación en cada régimen,
  el segundo VAR descansa mucho sobre el prior, y la verosimilitud marginal cobra ese precio. El
  TVAR se lee entonces como un **escenario de estrés** frente al modelo lineal, no como el único
  modelo de los datos.

**¿Qué cambia con la muestra y con el tratamiento?** El mismo modelo estimado hasta 2019 (sin
pandemia que tratar), con la muestra completa sin tratamiento, y con la alternativa al
escalamiento: sacar de la verosimilitud los trimestres 2020Q1 a 2021Q2 y los dos siguientes, cuyos
rezagos son la caída y el rebote (`covid_mode="dummies"`). Para cada uno, el umbral, el rezago, los
trimestres Niño, el factor de Bayes y la respuesta acumulada del PBI y de los ingresos a $+1$ °C en
cada régimen, que la sección 3 explica.

In [ ]:
def episodes(index):
    '''Trimestres consecutivos como rangos: 1998Q1-1999Q1, 2016Q1-2016Q4.'''
    out, start, prev = [], None, None
    for d in index:
        if start is None:
            start = prev = d
        elif (d.year - prev.year) * 4 + (d.quarter - prev.quarter) == 1:
            prev = d
        else:
            out.append((start, prev)); start = prev = d
    if start is not None:
        out.append((start, prev))
    return ", ".join(cd.quarter(a) if a == b else f"{cd.quarter(a)}-{cd.quarter(b)}" for a, b in out)

def cumulative(irf, m, var, reg, h=8):
    """Mediana de la respuesta acumulada entre el impacto y h trimestres a +1 °C."""
    if not irf.get(reg):
        return np.nan
    k, s = m.names.index(var), 0
    return float(np.median(irf[reg]["level"][:, :h + 1, k, s].sum(axis=1)))

def regime_row(m, irf):
    ts, p2, sp = m.threshold_summary(), m.regime_probability(), m.split_posterior
    row = {"T": m.T, "Z*": ts.loc["threshold (Z*)", "p50"],
           "d": int(sp.groupby("delay")["probability"].sum().idxmax()),
           "trimestres Niño": int((p2 > 0.5).sum()), "episodios": episodes(p2.index[p2 > 0.5]),
           "log BF (TVAR vs lineal)": m.log_marginal_likelihood()["log Bayes factor"]}
    for var, label in (("y", "PBI"), ("ing", "ingresos")):
        if var in m.names:
            row[f"{label} acum. 8, normal"] = cumulative(irf, m, var, "regime1")
            row[f"{label} acum. 8, Niño"] = cumulative(irf, m, var, "regime2")
    return row

def fit(data, block=True, threshold="nino12", **kw):
    n = data.shape[1]
    b = np.zeros((n, n), dtype=int)
    b[0, 1:] = 1
    opts = dict(threshold_var=threshold, lags=2, max_delay=4, post_draws=4000, burnin=0.5,
                hor=12, irf_1std=0, seed=1)
    opts.update(kw)
    m = ThresholdVAR(data, b_exo=b if block else None, **opts)
    m.sample_posterior(show_progress=False)
    return m, m.compute_irfs(shock="level", n_draws=200, reps=25, n_jobs=N_JOBS, scale=1.0, show_progress=False)

tvar19, irf19 = fit(Y.loc[:"2019-12-01"])
raw26, irf_raw = fit(Y)
dum26, irf_dum = fit(Y, covid_mode="dummies", covid_window=("2020Q1", "2021Q2"))
irf1 = model.compute_irfs(shock="level", n_draws=200, reps=25, n_jobs=N_JOBS, scale=1.0)
rows = {"1996Q1-2019Q4, sin la pandemia": regime_row(tvar19, irf19),
        "1996Q1-2026Q2, sin tratamiento": regime_row(raw26, irf_raw),
        "1996Q1-2026Q2, dummies 2020Q1-2021Q2 (+ rezagos)": regime_row(dum26, irf_dum),
        "1996Q1-2026Q2, Lenza-Primiceri 2020Q1-2021Q4 (base)": regime_row(model, irf1)}
pd.set_option("display.max_colwidth", 60)
pd.DataFrame(rows).T.round(2)

Tres lecturas. Hasta 2019 el modelo es parecido: umbral de 1.1 °C, rezago de cuatro trimestres y
los mismos episodios, corridos un trimestre. Sin tratamiento, el "régimen Niño" pasa a ser el
régimen de varianza alta: el umbral cae por debajo de cero, 69 trimestres caen en él, el factor de
Bayes se dispara a $+20$ y la asimetría del PBI desaparece. **Un factor de Bayes alto no garantiza
que el régimen sea el que uno cree.** Y las dos formas de tratar la pandemia, ponderar o excluir,
llegan al mismo sitio: cuando las escalas son grandes donde hace falta, dividir una fila por diez
es casi lo mismo que borrarla. Se usa la de Lenza y Primiceri porque mantiene las observaciones y
tiene una justificación en la literatura.

## 3. Respuestas por régimen

En un modelo no lineal la impulso-respuesta depende de la historia, del régimen y del tamaño y
signo del choque. `compute_irfs` calcula **respuestas generalizadas** (Koop, Pesaran y Potter,
1996): para cada extracción de la posterior simula la economía con y sin el choque desde cada
trimestre de un régimen, con las mismas innovaciones futuras, y deja que el régimen cambie a lo
largo de la trayectoria cuando la anomalía simulada cruza el umbral. Con `irf_1std=0` el choque
eleva la anomalía en exactamente 1 °C al impacto. Con exogeneidad de bloque, el choque llega a las
variables domésticas en el mismo trimestre a través de $\gamma_{i,0}$, el coeficiente del régimen
en que está cada trayectoria.

In [ ]:
model.plot_irfs(shock="nino12", response_vars=["y", "pi", "ing"], cred=0.68,
                regime_labels=("Normal", "Niño"), series_titles=titles[1:], ncol=3,
                title="Respuesta a un calentamiento de 1 °C en la región Niño 1+2")

In [ ]:
def response_table(irf, var, horizons=(0, 2, 4, 8)):
    v = names.index(var)
    rows = {}
    for reg, label in (("regime1", "normal"), ("regime2", "Niño")):
        a = irf[reg]["level"][:, :, v, 0]
        med = np.median(a, axis=0)
        rows[label] = {**{f"h = {h}": med[h] for h in horizons},
                       "acumulado a 8": np.median(a[:, :9].sum(axis=1))}
    return pd.DataFrame(rows).T

print("Respuesta mediana a +1 °C, en puntos porcentuales de la tasa interanual")
pd.concat({"PBI": response_table(irf1, "y"), "inflación": response_table(irf1, "pi"),
           "ingresos fiscales": response_table(irf1, "ing")}).round(2)

La asimetría es grande y aparece **desde el impacto**. En tiempos normales un grado más de
temperatura le resta una décima al crecimiento del PBI en el trimestre y unas siete décimas un año
después; durante un Niño, nueve décimas de inmediato y más de un punto durante el primer año: la
respuesta acumulada en dos años duplica la del régimen normal ($-9.2$ contra $-4.1$). La inflación
sube cuatro décimas al impacto durante un Niño (alimentos y transporte) y el efecto se disipa en un
año.

Los **ingresos fiscales** son la variable más sensible. En un Niño caen 1.8 puntos al impacto y la
respuesta acumulada en dos años, $-17.8$, es el doble de la del PBI: la caída de la recaudación
supera a la de la actividad que la genera. En tiempos normales la respuesta es pequeña y sus bandas
incluyen cero. Aplicar a un escenario de Niño una elasticidad unitaria entre ingresos y PBI
subestimaría la pérdida fiscal.

**Con y sin bloque.** ¿Qué hace la restricción? Se compara con el TVAR sin bloque, en el que la
anomalía también tiene dos regímenes y responde, con rezago, a la economía.

In [ ]:
irf_joint = joint.compute_irfs(shock="level", n_draws=200, reps=25, n_jobs=N_JOBS, scale=1.0, show_progress=False)
comparison = {}
for label, m, irf in (("con bloque", model, irf1), ("sin bloque", joint, irf_joint)):
    for var, name in (("y", "PBI"), ("ing", "ingresos")):
        a = irf["regime2"]["level"][:, :, names.index(var), 0]
        comparison[(name, label)] = {"impacto, Niño": np.median(a[:, 0]),
                                     "acumulado a 8, Niño": np.median(a[:, :9].sum(axis=1)),
                                     "acumulado a 8, normal": cumulative(irf, m, var, "regime1")}
print("Respuesta a +1 °C con y sin exogeneidad de bloque (medianas, pp)")
pd.DataFrame(comparison).T.round(2)

La restricción cambia el **perfil**, no tanto el tamaño. Sin bloque, el PBI cae el doble al impacto
durante un Niño ($-1.9$ contra $-0.9$), pero la respuesta acumulada en dos años es casi la misma
($-8.5$ contra $-9.2$): con el bloque, el choque se reparte en más trimestres. Los ingresos caen
más al impacto sin bloque y menos en el acumulado. Que una restricción que los datos favorecen
mueva la respuesta al impacto en un punto es un recordatorio: con trece trimestres en un régimen,
la parametrización importa. Lo robusto es el signo, la asimetría y el orden de magnitud acumulado.

**Tamaño y signo.** El modelo es lineal dentro de cada régimen, así que un choque de 3 °C es tres
veces el de 1 °C mientras la economía no cambie de régimen. Lo que puede cambiar el régimen es el
propio choque: un calentamiento grande en tiempos normales cruza el umbral y, tres trimestres
después, activa la dinámica Niño. Un enfriamiento (La Niña) es el espejo del calentamiento dentro
del régimen.

In [ ]:
irf3 = model.compute_irfs(shock="level", n_draws=200, reps=25, n_jobs=N_JOBS, scale=3.0)
irfm = model.compute_irfs(shock="level", n_draws=200, reps=25, n_jobs=N_JOBS, scale=-1.0)
size = pd.DataFrame({
    "+1 °C": response_table(irf1, "y")["acumulado a 8"],
    "+3 °C": response_table(irf3, "y")["acumulado a 8"],
    "-1 °C (La Niña)": response_table(irfm, "y")["acumulado a 8"]})
size["(+3) / (+1)"] = size["+3 °C"] / size["+1 °C"]
print("PBI, respuesta acumulada a 8 trimestres")
size.round(2)

En el régimen Niño la razón entre $+3$ °C y $+1$ °C es casi tres, como en un modelo lineal. En el
régimen normal es mayor, 3.7: parte de las simulaciones con $+3$ °C terminan en el régimen Niño.
Con $-1$ °C las respuestas cambian de signo y son algo menores que con $+1$ °C: el PBI acelera.

## 4. Escenarios: el Niño de 2026

En octubre de 2026 la anomalía Niño 1+2 ya superó los 3 °C en el promedio del tercer trimestre.
Un modelo con regímenes responde a una pregunta que el VAR lineal no puede: **¿cuándo y cuánto**
se sentirá en el PBI, en la inflación y en los ingresos fiscales? La densidad predictiva se simula
**a mano**, como en la sesión 2, con dos diferencias: la anomalía se impone en su propia ecuación,
que es exógena, y en cada paso la simulación consulta la anomalía rezagada para decidir con qué
régimen avanza el bloque doméstico.

Las condiciones iniciales son las observadas hasta 2026Q2 (el último dato macro) y la anomalía de
2026Q3, ya conocida. Los parámetros son los del modelo base, estimado con toda la muestra, así que
el Niño de 2023-24 ya está en el régimen.

In [ ]:
def simulate_block(parameters, history, h, imposed=None, seed=0):
    '''Sendas futuras del TVAR con exogeneidad de bloque. `parameters` es una lista, una entrada
    por extracción: {"Bx", "Cx", "B": (B1, B2), "C": (chol1, chol2), "tar", "delay"}. El bloque
    externo (la anomalía, columna 0) es lineal, con filas [rezago 1, rezago 2, constante]; el
    doméstico tiene filas [rezagos domésticos 1 y 2, constante, anomalía en t, rezagos 1 y 2 de la
    anomalía] y cambia de régimen con la anomalía rezagada. `imposed` fija la anomalía (NaN deja
    el paso libre); el resto de choques futuros se sortea.'''
    rng = np.random.default_rng(seed)
    hist = history.to_numpy()
    n, L, pre = hist.shape[1], 2, hist.shape[0]
    imposed = np.full(h, np.nan) if imposed is None else np.asarray(imposed, float)
    paths = np.empty((len(parameters), h, n))
    regime = np.empty((len(parameters), h), dtype=int)
    for m, par in enumerate(parameters):
        path = np.vstack([hist, np.zeros((h, n))])
        for j in range(h):
            t = pre + j
            s = 0 if path[t - par["delay"], 0] <= par["tar"] else 1       # régimen activo
            f_lags = np.array([path[t - l, 0] for l in range(1, L + 1)])
            f_mean = f_lags @ par["Bx"][:L, 0] + par["Bx"][L, 0]
            eps = rng.standard_normal(n)
            f = imposed[j] if not np.isnan(imposed[j]) else f_mean + par["Cx"][0, 0] * eps[0]
            x_lags = np.concatenate([path[t - l, 1:] for l in range(1, L + 1)])
            z = np.concatenate([x_lags, [1.0, f], f_lags])
            path[t] = np.r_[f, z @ par["B"][s] + par["C"][s] @ eps[1:]]
            regime[m, j] = s
        paths[m] = path[pre:]
    return paths, regime

def tvar_parameters(model, n_draws=1500):
    d = model.draws
    sel = np.linspace(0, d["tar"].shape[0] - 1, n_draws).astype(int)
    return [{"Bx": d["Bx"][i], "Cx": np.linalg.cholesky(d["Sx"][i]),
             "B": (d["B1"][i], d["B2"][i]),
             "C": (np.linalg.cholesky(d["S1"][i]), np.linalg.cholesky(d["S2"][i])),
             "tar": d["tar"][i], "delay": int(d["delay"][i])} for i in sel]

h = 8
history = Y.iloc[-4:]                                           # cuatro trimestres: rezagos y delay
dates = pd.date_range(history.index[-1] + pd.DateOffset(months=3), periods=h, freq="3MS")
nino_2026Q3 = levels.loc["2026-09-01", "nino12"]
print("Condiciones iniciales:"); display(history.round(2))
print(f"Anomalía observada en 2026Q3: {nino_2026Q3:.2f} °C; horizonte {cd.quarter(dates[0])} a {cd.quarter(dates[-1])}")

Cuatro escenarios para la anomalía desde 2026Q4. El primero copia la trayectoria del Niño de
1997-98 desde su pico, el episodio más parecido al actual; el segundo supone una disipación rápida;
el tercero no impone nada más y deja que la ecuación de la anomalía la proyecte. El cuarto es un
**contrafactual**: el mar nunca se calentó (anomalía cero desde 2026Q1), y sirve para medir el
costo del Niño como la diferencia con los otros tres.

In [ ]:
como_1998 = levels["nino12"].loc["1997-12-01":"1999-06-01"].to_numpy()      # 1997Q4 a 1999Q2
libre = np.full(h - 1, np.nan)
scenarios = {"como 1997-98": np.r_[nino_2026Q3, como_1998[:h - 1]],
             "disipación rápida": np.r_[nino_2026Q3, 2.0, 0.8, 0.0, 0.0, 0.0, 0.0, 0.0],
             "sin condicionar": np.r_[nino_2026Q3, libre],
             "sin Niño (contrafactual)": np.zeros(h)}
history_cf = history.copy()
history_cf.loc["2026-03-01":, "nino12"] = 0.0           # como si el mar no se hubiera calentado
pars = tvar_parameters(model)
paths, regimes = {}, {}
for name, path in scenarios.items():
    hist = history_cf if name.startswith("sin Niño") else history
    paths[name], regimes[name] = simulate_block(pars, hist, h, path)

labels = [cd.quarter(d) for d in dates]
print("Senda impuesta de la anomalía (°C):")
display(pd.DataFrame(scenarios, index=labels).T.round(2))
print("Probabilidad del régimen Niño en cada trimestre del horizonte:")
pd.DataFrame({k: v.mean(axis=0) for k, v in regimes.items()}, index=labels).T.round(2)

El régimen de 2027 ya está decidido por lo observado: con un rezago de tres trimestres, las
anomalías de 2026Q2 y 2026Q3 ponen a la economía en el régimen Niño desde el primer trimestre de
2027 en cualquier escenario con el calentamiento. Los escenarios difieren en cuánto dura. La
probabilidad de 0.14 en los trimestres sin calentamiento es la moda secundaria del umbral.

In [ ]:
colors = {"como 1997-98": "#B0413E", "disipación rápida": "#E78033", "sin condicionar": "#1F3D5C",
          "sin Niño (contrafactual)": "#7F7F7F"}
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for ax, var, title in zip(axes, ["y", "pi", "ing"], titles[1:]):
    k = names.index(var)
    plot_fan_chart(Y[var], paths["como 1997-98"][:, :, k], dates, bands=(0.90, 0.68),
                   history_from="2022-03-01", hline=0, ax=ax, color=colors["como 1997-98"],
                   center_label="mediana, como 1997-98", history_label="observado",
                   title=f"{title}: Niño de 2026")
    for name in ("disipación rápida", "sin condicionar", "sin Niño (contrafactual)"):
        ax.plot(dates, np.median(paths[name][:, :, k], axis=0), color=colors[name], lw=1.8,
                ls="--" if name.startswith("sin Niño") else "-", label=f"mediana, {name}")
    ax.legend(fontsize=7, loc="lower left")
plt.show()

In [ ]:
def yearly(paths_dict, var, year):
    k = names.index(var)
    cols = [j for j, d in enumerate(dates) if d.year == year]
    return {name: np.percentile(p[:, cols, k].mean(axis=1), [16, 50, 84]) for name, p in paths_dict.items()}

summary = {}
for var, label in (("y", "PBI 2027"), ("pi", "inflación 2027"), ("ing", "ingresos fiscales 2027")):
    for name, q in yearly(paths, var, 2027).items():
        summary[(label, name)] = {"p16": q[0], "mediana": q[1], "p84": q[2]}
print("Promedio de 2027, en var. % interanual (percentiles de las trayectorias)")
pd.DataFrame(summary).T.round(1)

Un Niño que repita 1997-98 deja el crecimiento de 2027 en $-0.5$ %, contra 3.7 % en el
contrafactual: **cuatro puntos de PBI**, con el golpe concentrado en la primera mitad del año (la
mediana cae a $-1.7$ % interanual en 2027Q2) y una recuperación rápida después. Los ingresos
fiscales crecen cero en 2027, contra 5.6 % sin Niño. Si el evento se disipa pronto, el costo en el
PBI baja a un punto. La inflación de 2027 sube a 4.5 % en el escenario como 1997-98, contra 3.7 %
sin Niño, pero las bandas son anchas.

**El VAR lineal, para comparar.** Un `BayesianVAR` con los mismos datos, rezagos, tratamiento de
la pandemia y exogeneidad de bloque (en `BayesianVAR` la restricción es un prior muy ajustado sobre
los rezagos excluidos) no conoce regímenes: para él, el Niño es un choque grande a una variable y
nada más. Se simula con la misma lógica y las mismas sendas de la anomalía.

In [ ]:
def simulate_linear(parameters, history, h, imposed, seed=0):
    '''Sendas de un VAR lineal (un solo régimen) con la anomalía primera en el orden recursivo:
    la anomalía impuesta se obtiene con su propio choque estructural.'''
    rng = np.random.default_rng(seed)
    hist = history.to_numpy()
    n, L, pre = hist.shape[1], 2, hist.shape[0]
    imposed = np.asarray(imposed, float)
    paths = np.empty((len(parameters), h, n))
    for m, par in enumerate(parameters):
        path = np.vstack([hist, np.zeros((h, n))])
        for j in range(h):
            t = pre + j
            lags = np.concatenate([path[t - l] for l in range(1, L + 1)])
            mean = lags @ par["B"][:-1] + par["B"][-1]
            eps = rng.standard_normal(n)
            if not np.isnan(imposed[j]):
                eps[0] = (imposed[j] - mean[0]) / par["C"][0, 0]
            path[t] = mean + par["C"] @ eps
        paths[m] = path[pre:]
    return paths

bvar = BayesianVAR(Y, lags=2, prior_type=1, post_draws=4000, burnin=0.5, seed=1, progress=False,
                   b_exo=b_exo, covid_mode="lenza-primiceri", covid_window=window, covid_n_free=8)
bvar.sample_posterior()

def linear_parameters(bvar, n_draws=1500):
    beta, sigma = np.asarray(bvar.beta_draws), np.asarray(bvar.Sigma_draws)
    n, k = sigma.shape[1], bvar.ncoeff_eq
    sel = np.linspace(0, beta.shape[0] - 1, n_draws).astype(int)
    out = []
    for i in sel:
        coef = beta[i].reshape(n, k)                 # fila: ecuación, constante al final
        out.append({"B": np.vstack([coef[:, :-1].T, coef[:, -1]]), "C": np.linalg.cholesky(sigma[i])})
    return out

pars_lin = linear_parameters(bvar)
paths_lin = {name: simulate_linear(pars_lin, history_cf if name.startswith("sin Niño") else history,
                                   h, path) for name, path in scenarios.items()}
rows = {}
for var, label in (("y", "PBI"), ("ing", "ingresos fiscales")):
    for model_label, p in (("TVAR", paths), ("VAR lineal", paths_lin)):
        med = {k: v[1] for k, v in yearly(p, var, 2027).items()}
        base_cf = med.pop("sin Niño (contrafactual)")
        rows[(label, model_label)] = {**{f"2027, {k}": v for k, v in med.items()},
                                      "contrafactual": base_cf,
                                      "costo, como 1997-98": base_cf - med["como 1997-98"]}
print("Promedio de 2027 (mediana, var. % interanual) y costo del Niño como 1997-98 (pp)")
display(pd.DataFrame(rows).T.round(1))
k = names.index("y")
print("Trimestre a trimestre, escenario como 1997-98 (mediana del crecimiento del PBI)")
pd.DataFrame({"TVAR": np.median(paths["como 1997-98"][:, :, k], axis=0),
              "VAR lineal": np.median(paths_lin["como 1997-98"][:, :, k], axis=0)}, index=labels).T.round(1)

El TVAR duplica el costo del VAR lineal en un Niño como el de 1997-98: cuatro puntos de crecimiento
del PBI en 2027 contra dos, y cinco puntos y medio de crecimiento de los ingresos contra dos. Y
cambia su forma: el TVAR concentra el golpe en 2027Q2 y la economía se recupera en cuanto sale del
régimen; el VAR lineal reparte un golpe menor durante todo el año. Las dos conclusiones hay que
tomarlas con pinzas: el régimen Niño está estimado con trece trimestres, el factor de Bayes no
favorece al TVAR y el modelo no sabe nada de la respuesta fiscal ni de la reconstrucción que siguió
a cada Niño. Por eso el TVAR sirve para dimensionar un **escenario de estrés**, no para reemplazar
al modelo lineal.

## 5. Robustez

¿Dependen el régimen y la asimetría de la elección del sistema? Se reestima el TVAR, siempre con
las escalas de Lenza y Primiceri: sin exogeneidad de bloque, con los términos de intercambio como
control de los choques externos (para que el régimen Niño no absorba la crisis asiática de 1998 ni
el fin del superciclo en 2016), sin ingresos y desde 1994 (el sistema de versiones anteriores de
este notebook) y con el índice Niño 3.4 en lugar del 1+2.

In [ ]:
base = levels[["nino12", "nino34"]].join(g).join(cd.yoy(levels[["tdi"]]).rename(columns={"tdi": "tot"}))

def fit_spec(cols, threshold="nino12", start="1996-03-01", block=True, window=("2020Q1", "2021Q4"), **kw):
    df = base[cols].loc[start:"2026-06-01"].dropna()
    m, irf = fit(df, block=block, threshold=threshold, covid_mode="lenza-primiceri",
                 covid_window=window, covid_n_free=8, **kw)
    row = regime_row(m, irf)
    row["muestra"] = f"{cd.quarter(df.index[0])}-{cd.quarter(df.index[-1])}"
    return row

specs = {
    "base: y, pi, ing":                ({"cols": ["nino12", "y", "pi", "ing"]}),
    "sin exogeneidad de bloque":       ({"cols": ["nino12", "y", "pi", "ing"], "block": False}),
    "control: términos de intercambio": ({"cols": ["nino12", "tot", "y", "pi", "ing"]}),
    "sin ingresos: y, pi desde 1994":  ({"cols": ["nino12", "y", "pi"], "start": "1994-03-01"}),
    "umbral Niño 3.4":                 ({"cols": ["nino34", "y", "pi", "ing"], "threshold": "nino34"}),
}
table = pd.DataFrame({label: fit_spec(**args) for label, args in specs.items()}).T
cols = ["muestra", "T", "Z*", "d", "trimestres Niño", "episodios", "log BF (TVAR vs lineal)",
        "PBI acum. 8, normal", "PBI acum. 8, Niño", "ingresos acum. 8, normal", "ingresos acum. 8, Niño"]
table[cols].round(2)

Tres lecturas.

* **El régimen es robusto** a las variables del sistema: con o sin bloque, con los términos de
  intercambio o sin ingresos, los episodios son los mismos (1998-99, 2016 y 2024, a lo sumo
  corridos un trimestre) y el umbral está entre 1.2 y 1.5 °C. La respuesta acumulada del PBI en el
  régimen Niño es entre 1.7 y 3.5 veces la del normal, y la de los ingresos, entre 1.5 y 2 veces la
  del PBI.
* **El factor de Bayes del umbral es negativo en todas las especificaciones**: el régimen está
  bien identificado en el tiempo, pero la verosimilitud marginal no premia sus coeficientes
  adicionales.
* **Es la región 1+2, no la 3.4.** Con el índice Niño 3.4, el del Pacífico central que define los
  eventos globales, el régimen mezcla 1998 y 2016 con 2003Q4 y 2010-11, la asimetría del PBI
  desaparece y la respuesta de los ingresos cambia de signo: lo que afecta a Perú es el
  calentamiento frente a su costa.

## Ideas centrales

* **La temperatura del mar es exógena, y los datos lo confirman.** Con exogeneidad de bloque la
  anomalía es un AR lineal y la economía responde a ella; el factor de Bayes favorece la
  restricción (+3.2).
* **El régimen es el año siguiente al calentamiento.** El umbral está cerca de 1.2 °C y el rezago es
  de tres trimestres: trece trimestres caen en el régimen Niño (1998-99, 2016, 2017Q4 y
  2024Q1-Q3). El rezago es la regla de clasificación, no el plazo del daño.
* **La evidencia formal es limitada.** El log factor de Bayes del umbral es $-5.2$: con pocos
  trimestres en el segundo régimen, el prior pesa. Y un factor de Bayes alto tampoco basta: sin
  tratar la pandemia, el segundo régimen pasa a ser el de varianza alta.
* **El Niño golpea más a los ingresos fiscales que al PBI.** Un calentamiento de 1 °C durante un
  Niño resta nueve décimas de crecimiento al PBI al impacto y 9.2 puntos acumulados en dos años;
  a los ingresos, 1.8 al impacto y 17.8 acumulados.
* **La restricción cambia el perfil.** Sin bloque, la caída al impacto del PBI se duplica, pero la
  acumulada es parecida.
* **Los escenarios dimensionan riesgos.** Un Niño como el de 1997-98 deja el crecimiento de 2027
  en $-0.5$ %, cuatro puntos menos que el contrafactual sin Niño, y los ingresos sin crecer; el VAR
  lineal da la mitad. Son escenarios condicionales a trayectorias impuestas de la temperatura.

## Ejercicios

1. **El prior de la anomalía corriente.** Reestime el modelo base con
   `prior_params={"lamda0P": 100}` (un prior cien veces más laxo sobre $\gamma_{i,0}$) y con
   `prior_params={"lamdaP": 0.5}` (rezagos más laxos). ¿Cuál de los dos mueve la respuesta al
   impacto del PBI en el régimen Niño? ¿Se acerca a la del modelo sin bloque?
2. **Umbral exógeno.** Sánchez y Vassallo (2026) fijan el umbral de confianza en 50 puntos en lugar
   de estimarlo. Fije `threshold=1.0`, el límite del Niño moderado en el índice ICEN del ENFEN, y
   compare los regímenes, el factor de Bayes y las respuestas con los de la sección 2. ¿Y con
   0.4 °C, el límite del Niño débil?
3. **Escenarios alternativos.** Repita la sección 4 con la senda del Niño de 2015-16 y con la de
   2023. ¿En qué trimestre de 2027 caen más el PBI y los ingresos en cada caso, y por qué?
4. **La moda secundaria.** Separe las extracciones con umbral menor a 0.5 °C y calcule las
   respuestas solo con las demás: filtre las claves de `model.draws` que tienen una fila por
   extracción antes de llamar a `compute_irfs`. ¿Cuánto cambian las respuestas del régimen Niño?
5. **Choques financieros.** Reemplace la anomalía por el VIX como variable umbral en el sistema
   macrofiscal `(vix, x, y, r)` de las sesiones 1 y 2, con el VIX como bloque externo. ¿Aparecen
   regímenes de estrés financiero global? ¿Qué dice el factor de Bayes?

## Referencias

- Aguirre, J., Ledesma, A., Pérez, F. y Rojas, Y. (2025). El Niño Costero y la economía peruana: efectos sobre actividad e inflación. Banco Central de Reserva del Perú, Documento de Trabajo 2025-008.
- Alessandri, P. y Mumtaz, H. (2017). Financial conditions and density forecasts for US output and inflation. *Review of Economic Dynamics* 24, 66-78.
- Alessandri, P. y Mumtaz, H. (2019). Financial regimes and uncertainty shocks. *Journal of Monetary Economics* 101, 31-46.
- Bańbura, M., Giannone, D. y Reichlin, L. (2010). Large Bayesian vector auto regressions. *Journal of Applied Econometrics* 25(1), 71-92.
- Chen, C.W.S. y Lee, J.C. (1995). Bayesian inference of threshold autoregressive models. *Journal of Time Series Analysis* 16(5), 483-492.
- Cushman, D.O. y Zha, T. (1997). Identifying monetary policy in a small open economy under flexible exchange rates. *Journal of Monetary Economics* 39(3), 433-448.
- Koop, G., Pesaran, M.H. y Potter, S.M. (1996). Impulse response analysis in nonlinear multivariate models. *Journal of Econometrics* 74(1), 119-147.
- Lenza, M. y Primiceri, G.E. (2022). How to estimate a vector autoregression after March 2020. *Journal of Applied Econometrics* 37(4), 688-699.
- Sánchez, W. y Vassallo, R. (2026). ¿Depende la transmisión de los choques externos del estado de la confianza empresarial? Evidencia para el Perú, 2002-2025. Versión revisada del Documento de Investigación 01-2026, Consejo Fiscal del Perú.
- NOAA Physical Sciences Laboratory. Niño 1+2 and Niño 3.4 SST anomalies (ERSST v6), https://psl.noaa.gov/data/climateindices/.